# Solución · Lab 02 · Lectura de datos y esquemas

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crapdev/sparkLab/blob/main/labs/02-lectura-esquemas/solution.ipynb)

**Objetivo:** Leer archivos CSV y JSON controlando los tipos de las columnas, con esquemas inferidos y con esquemas escritos a mano.

**Tiempo estimado:** 35 minutos

> Esta es la solución completa. Intenta resolver el laboratorio por tu cuenta antes
> de leerla: equivocarse y corregir es donde más se aprende.

## Preparación

Esta celda instala la versión de PySpark con la que se probó el laboratorio y comprueba
que hay una versión de Java compatible. En Colab tarda alrededor de un minuto la primera vez.

In [ ]:
import importlib.metadata
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.2.0"
MIN_JAVA_VERSION = 17


def installed_pyspark_version():
    try:
        return importlib.metadata.version("pyspark")
    except importlib.metadata.PackageNotFoundError:
        return None


def java_major_version():
    java = shutil.which("java")
    if java is None:
        return 0
    output = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
    match = re.search(r'version "(\d+)', output)
    return int(match.group(1)) if match else 0


if installed_pyspark_version() != PYSPARK_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )

if java_major_version() < MIN_JAVA_VERSION:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Spark 4 necesita Java 17 o superior. Instálalo y repite esta celda.")
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(f"PySpark {PYSPARK_VERSION} y Java {java_major_version()} listos.")

La siguiente celda define `check`, la función que usan los tests. No hace falta que la
modifiques: ejecútala y sigue.

In [ ]:
RESULTS = {}


def check(name, condition, hint):
    """Run one test: `condition` is a function that must return True."""
    try:
        passed = bool(condition())
        error = None
    except Exception as exc:  # A broken answer is a failed test, not a crash.
        passed, error = False, exc
    RESULTS[name] = passed
    if passed:
        print(f"✅ {name}")
        return
    detail = f" ({type(error).__name__}: {error})" if error else ""
    print(f"❌ {name}{detail}")
    print(f"   Pista: {hint}")
    assert passed, f"{name} · {hint}"

## Sesión de Spark

Esta celda crea la `SparkSession` del laboratorio e importa las funciones como `F`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("lab-02-lectura-esquemas").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo.")

In [ ]:
import time


def measure(action):
    """Run `action` and report the jobs, stages and tasks Spark launched for it."""
    tracker = spark.sparkContext.statusTracker()
    jobs_before = set(tracker.getJobIdsForGroup())
    result = action()
    time.sleep(0.3)  # Give Spark's listener a moment to register finished jobs.
    job_ids = sorted(set(tracker.getJobIdsForGroup()) - jobs_before)
    stage_ids = [stage for job in job_ids for stage in tracker.getJobInfo(job).stageIds]
    stages = [tracker.getStageInfo(stage) for stage in stage_ids]
    return {
        "result": result,
        "jobs": len(job_ids),
        "stages": len(stage_ids),
        "tasks": sum(stage.numTasks for stage in stages if stage is not None),
    }


def report(label, measurement):
    """Print one measurement in a single readable line."""
    print(
        f"{label:<28} jobs={measurement['jobs']}  "
        f"stages={measurement['stages']}  tasks={measurement['tasks']}"
    )

## Datos

Esta celda descarga `bigmart_sales.csv`, `drivers.json`, `drivers_multiline.json` la primera vez y los deja en la carpeta `data/`.
Usa `dataset_path("nombre")` para obtener la ruta de un archivo.

In [ ]:
import os
import urllib.request

DATA_BASE_URLS = [
    "https://raw.githubusercontent.com/crapdev/sparkLab/main/public/data",
]
DATA_DIR = os.environ.get("PYSPARK_LABS_DATA_DIR", "data")
DATASETS = ["bigmart_sales.csv", "drivers.json", "drivers_multiline.json"]


def dataset_path(name):
    """Return the local path of a dataset, downloading it the first time."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    os.makedirs(DATA_DIR, exist_ok=True)
    for base_url in DATA_BASE_URLS:
        try:
            urllib.request.urlretrieve(f"{base_url}/{name}", path)
            return path
        except OSError:
            continue  # Try the next mirror.
    raise RuntimeError(f"No se pudo descargar {name}. Revisa tu conexión y repite la celda.")


for dataset in DATASETS:
    print("Listo:", dataset_path(dataset))

In [ ]:
from pyspark.sql.types import DateType, IntegerType, StringType, StructField, StructType

csv_path = dataset_path("bigmart_sales.csv")
drivers_path = dataset_path("drivers.json")
drivers_multiline_path = dataset_path("drivers_multiline.json")

## Antes de empezar · qué pasa si no le dices nada a Spark

Lee el CSV sin ninguna opción y mira el resultado. Spark no sabe que la primera fila son
los nombres de las columnas, así que las llama `_c0`, `_c1`... y trata la cabecera como
un dato más. Además, todas las columnas son texto.

In [ ]:
raw = spark.read.format("csv").load(csv_path)

print("Filas:", raw.count())
print("Tipos distintos:", {column_type for _, column_type in raw.dtypes})
raw.select("_c0", "_c1", "_c2").show(3)

## Tarea 1 · Deja que Spark deduzca el esquema

Lee `csv_path` en `sales_inferred` de forma que:

- la primera fila se use como nombres de columna
- Spark deduzca el tipo de cada columna

In [ ]:
sales_inferred = (
    spark.read.format("csv").option("header", True).option("inferSchema", True).load(csv_path)
)

In [ ]:
check(
    "Tarea 1 · 8523 filas y 12 columnas",
    lambda: sales_inferred.count() == 8523 and len(sales_inferred.columns) == 12,
    'Falta .option("header", True): sin ella la cabecera cuenta como una fila más.',
)
check(
    "Tarea 1 · las columnas tienen su nombre",
    lambda: sales_inferred.columns[0] == "Item_Identifier",
    'Con .option("header", True) la primera fila pasa a ser los nombres.',
)
check(
    "Tarea 1 · los tipos se han deducido",
    lambda: dict(sales_inferred.dtypes)["Item_Weight"] == "double"
    and dict(sales_inferred.dtypes)["Outlet_Establishment_Year"] == "int",
    'Falta .option("inferSchema", True): sin ella todas las columnas son string.',
)

`printSchema()` es la forma más cómoda de ver el esquema. Fíjate en los tipos: te harán
falta en la tarea siguiente.

In [ ]:
sales_inferred.printSchema()

## Tarea 2 · Escribe el esquema como texto DDL

`inferSchema` es cómodo, pero obliga a Spark a recorrer el archivo entero solo para
adivinar los tipos. Si ya los conoces, es mejor dárselos.

La forma más corta es un **string DDL**: `"nombre TIPO, nombre TIPO, ..."`, igual que en
un `CREATE TABLE` de SQL.

1. Completa `SALES_DDL` con las 12 columnas, en el mismo orden que el archivo y con los
   mismos tipos que dedujo Spark. Los tipos que necesitas son `STRING`, `DOUBLE` e `INT`.
2. Lee el CSV en `sales_ddl` usando `.schema(SALES_DDL)` en lugar de `inferSchema`.
   La opción `header` sigue haciendo falta: sin ella la cabecera se leería como datos.

In [ ]:
SALES_DDL = """
    Item_Identifier STRING,
    Item_Weight DOUBLE,
    Item_Fat_Content STRING,
    Item_Visibility DOUBLE,
    Item_Type STRING,
    Item_MRP DOUBLE,
    Outlet_Identifier STRING,
    Outlet_Establishment_Year INT,
    Outlet_Size STRING,
    Outlet_Location_Type STRING,
    Outlet_Type STRING,
    Item_Outlet_Sales DOUBLE
"""

sales_ddl = spark.read.format("csv").option("header", True).schema(SALES_DDL).load(csv_path)

In [ ]:
check(
    "Tarea 2 · el esquema tiene las 12 columnas en orden",
    lambda: sales_ddl.columns == sales_inferred.columns,
    "Revisa SALES_DDL: tienen que estar las 12 columnas, con el mismo nombre y orden que el archivo.",
)
check(
    "Tarea 2 · los tipos coinciden con los deducidos",
    lambda: sales_ddl.dtypes == sales_inferred.dtypes,
    "Algún tipo no coincide. Compara con el printSchema() de la tarea 1: STRING, DOUBLE o INT.",
)
check(
    "Tarea 2 · se leen las 8523 filas",
    lambda: sales_ddl.count() == 8523,
    'Mantén .option("header", True) aunque des el esquema.',
)
check(
    "Tarea 2 · los datos son los mismos",
    lambda: sales_ddl.exceptAll(sales_inferred).count() == 0,
    "Con un tipo equivocado Spark rellena con NULL. Revisa los tipos de SALES_DDL.",
)

Ahora mide la diferencia. Con el esquema escrito, leer no lanza ningún job: Spark no
abre el archivo hasta que una acción lo necesita.

In [ ]:
report(
    "header + inferSchema",
    measure(lambda: spark.read.format("csv").option("header", True).option("inferSchema", True).load(csv_path)),
)
report(
    "esquema DDL",
    measure(lambda: spark.read.format("csv").option("header", True).schema(SALES_DDL).load(csv_path)),
)

### Dos trampas de los esquemas en CSV

**1. El esquema se aplica por posición, no por nombre.** Spark no comprueba que tus
nombres coincidan con la cabecera. Si cambias el orden, los datos acaban bajo el nombre
equivocado y nadie avisa:

In [ ]:
swapped = spark.read.format("csv").option("header", True).schema("Item_Weight STRING, Item_Identifier STRING").load(csv_path)
swapped.show(3)

**2. Un tipo que no encaja se convierte en `NULL`.** Por defecto Spark no falla: deja
vacío lo que no puede convertir. Aquí se declara el identificador como número:

In [ ]:
wrong_type = spark.read.format("csv").option("header", True).schema("Item_Identifier INT, Item_Weight DOUBLE").load(csv_path)
wrong_type.show(3)
print("Identificadores nulos:", wrong_type.filter(F.col("Item_Identifier").isNull()).count(), "de", wrong_type.count())

## Tarea 3 · Lee un JSON

`drivers.json` tiene un piloto de Fórmula 1 por línea. Ese formato se llama **JSON Lines**
y es el que Spark espera por defecto.

Lee `drivers_path` en `drivers`. En JSON no hacen falta `header` ni `inferSchema`: los
nombres van dentro de cada registro y Spark siempre deduce los tipos.

In [ ]:
drivers = spark.read.format("json").load(drivers_path)

In [ ]:
check(
    "Tarea 3 · 885 pilotos",
    lambda: drivers.count() == 885,
    'Usa spark.read.format("json").load(drivers_path), sin más opciones.',
)
check(
    "Tarea 3 · 8 columnas",
    lambda: len(drivers.columns) == 8 and "driverRef" in drivers.columns,
    "Si solo ves la columna _corrupt_record, quita la opción multiLine: este archivo tiene un registro por línea.",
)
check(
    "Tarea 3 · name es un struct",
    lambda: dict(drivers.dtypes)["name"] == "struct<forename:string,surname:string>",
    "Deja que Spark deduzca el esquema: no le pases uno propio en esta tarea.",
)

Mira el esquema. Hay tres cosas que conviene notar:

- Las columnas salen en **orden alfabético**, no en el orden del archivo.
- `name` es un **struct**: una columna que contiene otras dos, `forename` y `surname`.
- `dob` (fecha de nacimiento) es `string`: Spark no deduce fechas en JSON.

In [ ]:
drivers.printSchema()
drivers.show(3, truncate=False)

## Tarea 4 · Saca los campos anidados

Para llegar a un campo dentro de un struct se usa un punto: `"name.forename"`.

Crea `driver_names` con exactamente estas tres columnas, en este orden:
`driverRef`, `forename` y `surname`.

In [ ]:
driver_names = drivers.select("driverRef", "name.forename", "name.surname")

In [ ]:
check(
    "Tarea 4 · columnas driverRef, forename y surname",
    lambda: driver_names.columns == ["driverRef", "forename", "surname"],
    'Usa .select("driverRef", "name.forename", "name.surname").',
)
check(
    "Tarea 4 · siguen los 885 pilotos",
    lambda: driver_names.count() == 885,
    "Solo hay que seleccionar columnas, sin filtrar filas.",
)
check(
    "Tarea 4 · los nombres son correctos",
    lambda: driver_names.filter(F.col("driverRef") == "hamilton").first()["forename"] == "Lewis",
    "forename tiene que salir de name.forename.",
)

## Tarea 5 · Un JSON de varias líneas

`drivers_multiline.json` contiene 20 pilotos, pero escritos como un único array con
sangría, que ocupa muchas líneas. Si lo lees como en la tarea 3, Spark intenta
interpretar cada línea como un registro completo y no entiende ninguna:

In [ ]:
broken = spark.read.format("json").load(drivers_multiline_path)
print("Columnas:", broken.columns)

Esa columna `_corrupt_record` es la señal de que Spark no pudo interpretar el archivo.

Lee `drivers_multiline_path` en `drivers_multi` indicándole a Spark que cada registro
puede ocupar varias líneas, con la opción `multiLine`.

In [ ]:
drivers_multi = spark.read.format("json").option("multiLine", True).load(drivers_multiline_path)

In [ ]:
check(
    "Tarea 5 · sin registros corruptos",
    lambda: "_corrupt_record" not in drivers_multi.columns,
    'Añade .option("multiLine", True).',
)
check(
    "Tarea 5 · 20 pilotos",
    lambda: drivers_multi.count() == 20,
    "Lee drivers_multiline_path, no drivers_path.",
)
check(
    "Tarea 5 · mismo esquema que el JSON de una línea",
    lambda: drivers_multi.dtypes == drivers.dtypes,
    "Deja que Spark deduzca el esquema.",
)

## Tarea 6 · Un esquema con StructType

El string DDL es corto, pero el esquema también se puede construir con objetos:
`StructType` es la tabla y cada `StructField(nombre, tipo)` es una columna. Es más largo
de escribir, y a cambio se puede generar y modificar con código.

Completa `DRIVERS_SCHEMA` con estas 8 columnas, en este orden:

| Columna | Tipo |
|---|---|
| `driverId` | `IntegerType()` |
| `driverRef` | `StringType()` |
| `number` | `IntegerType()` |
| `code` | `StringType()` |
| `name` | un `StructType` con `forename` y `surname`, los dos `StringType()` |
| `dob` | `DateType()` |
| `nationality` | `StringType()` |
| `url` | `StringType()` |

Después lee `drivers_path` en `drivers_typed` usando `.schema(DRIVERS_SCHEMA)`.

In [ ]:
DRIVERS_SCHEMA = StructType(
    [
        StructField("driverId", IntegerType()),
        StructField("driverRef", StringType()),
        StructField("number", IntegerType()),
        StructField("code", StringType()),
        StructField(
            "name",
            StructType(
                [
                    StructField("forename", StringType()),
                    StructField("surname", StringType()),
                ]
            ),
        ),
        StructField("dob", DateType()),
        StructField("nationality", StringType()),
        StructField("url", StringType()),
    ]
)

drivers_typed = spark.read.format("json").schema(DRIVERS_SCHEMA).load(drivers_path)

In [ ]:
check(
    "Tarea 6 · las 8 columnas en orden",
    lambda: DRIVERS_SCHEMA.fieldNames()
    == ["driverId", "driverRef", "number", "code", "name", "dob", "nationality", "url"],
    "Revisa los nombres y el orden de los StructField.",
)
check(
    "Tarea 6 · driverId y number son enteros",
    lambda: dict(drivers_typed.dtypes)["driverId"] == "int" and dict(drivers_typed.dtypes)["number"] == "int",
    "Usa IntegerType() para driverId y number.",
)
check(
    "Tarea 6 · dob es una fecha",
    lambda: dict(drivers_typed.dtypes)["dob"] == "date",
    "Usa DateType() para dob.",
)
check(
    "Tarea 6 · name sigue siendo un struct",
    lambda: dict(drivers_typed.dtypes)["name"] == "struct<forename:string,surname:string>",
    "name es un StructType con dos StructField: forename y surname.",
)
check(
    "Tarea 6 · se leen los 885 pilotos con sus datos",
    lambda: drivers_typed.count() == 885
    and str(drivers_typed.filter(F.col("driverRef") == "hamilton").first()["dob"]) == "1985-01-07",
    "Lee drivers_path con .schema(DRIVERS_SCHEMA). Si dob sale NULL, revisa el nombre de la columna.",
)

Compara con el esquema que dedujo Spark en la tarea 3: ahora las columnas están en el
orden que tú elegiste, los identificadores son `integer` y `dob` es una fecha de verdad.
Y, como en el CSV, leer con esquema no lanza ningún job.

In [ ]:
drivers_typed.printSchema()

report("JSON con esquema deducido", measure(lambda: spark.read.format("json").load(drivers_path)))
report("JSON con StructType", measure(lambda: spark.read.format("json").schema(DRIVERS_SCHEMA).load(drivers_path)))

Una diferencia importante con el CSV: en JSON las columnas se emparejan **por nombre**.
Si en el esquema pides un campo que no existe en el archivo, la columna sale entera
en `NULL`, sin ningún error:

In [ ]:
spark.read.format("json").schema("driverRef STRING, nickname STRING").load(drivers_path).show(3)

## Resumen

Ejecuta esta celda al final. Si todos los tests pasan, vuelve al sitio y marca el
laboratorio como completado.

In [ ]:
EXPECTED_TESTS = 21
passed_tests = sum(RESULTS.values())

print(f"{passed_tests} de {EXPECTED_TESTS} tests superados")
for test_name, passed in RESULTS.items():
    print("  ✅" if passed else "  ❌", test_name)

assert passed_tests == EXPECTED_TESTS, "Todavía quedan tareas por resolver."
print("\n🎉 ¡Laboratorio completado! Vuelve al sitio y márcalo como hecho.")